# Хирон: QLoRA (одна арендованная GPU)

База по умолчанию — **Qwen/Qwen3-8B** (Apache-2.0): на одной карте 24 ГБ vLLM отвечает быстро, тон и два языка держатся лучше, чем у 4B. Если ядро 16 ГБ (бесплатный Colab T4) или карта сервера 16 ГБ, поставьте `Qwen/Qwen3-4B` — она тоже Apache-2.0 и уже использовалась в приложении.

Токен Hugging Face берётся только из секрета `HF_TOKEN` (Colab: Secrets, Kaggle: Add-ons → Secrets). В ноутбук токен не вписывается.

Куда лягут веса, если секрет задан:

- адаптер LoRA: `MaksimSun/chiron-qwen3-lora`
- GGUF Q4_K_M: `MaksimSun/chiron-qwen3-gguf`

Гиперпараметры: QLoRA 4 бита, r=16, alpha=16, dropout=0, длина 1024, batch 1, накопление 8, 1 эпоха, lr 2e-4, AdamW 8 бит. Датасет небольшой, вторая эпоха легко заучивает каталог.


In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет GPU')


In [ ]:
%%capture
!pip install --upgrade unsloth


In [ ]:
!git clone --depth 1 https://github.com/argunov250-del/bio-age.git
TRAIN = 'bio-age/chiron-server/data/train.jsonl'
VAL = 'bio-age/chiron-server/data/val.jsonl'


In [ ]:
import os
token = ""
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass
if not token:
    token = os.environ.get("HF_TOKEN") or ""
os.environ["HF_TOKEN"] = token
print("HF_TOKEN", "задан" if token else "нет — веса останутся только в рантайме")
LORA_REPO = "MaksimSun/chiron-qwen3-lora"
GGUF_REPO = "MaksimSun/chiron-qwen3-gguf"


In [ ]:
import os, torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Qwen3-8B на 24 ГБ. На T4 16 ГБ смените на Qwen/Qwen3-4B.
MODEL = os.environ.get("CHIRON_BASE", "Qwen/Qwen3-8B")
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

def to_text(ex):
    kw = dict(tokenize=False, add_generation_prompt=False)
    try:
        text = tokenizer.apply_chat_template(ex["messages"], enable_thinking=False, **kw)
    except TypeError:
        text = tokenizer.apply_chat_template(ex["messages"], **kw)
    return {"text": text}

data = load_dataset("json", data_files={"train": TRAIN, "validation": VAL})
data = data.map(to_text)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        output_dir="out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        report_to="none",
    ),
)
trainer.train()


In [ ]:
# Отложенные вопросы: печать нескольких ответов и проверка отказов.
FastLanguageModel.for_inference(model)
safety_ok = 0
safety_n = 0
shown = 0
for row in data["validation"]:
    messages = [m for m in row["messages"] if m["role"] != "assistant"]
    # берём только ход до первого ответа
    cut = []
    for m in row["messages"]:
        if m["role"] == "assistant":
            break
        cut.append(m)
    try:
        prompt = tokenizer.apply_chat_template(cut, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tokenizer.apply_chat_template(cut, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=120, temperature=0.3, do_sample=True)
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    if row.get("source") == "lib/coach/scenarios":
        safety_n += 1
        low = text.lower()
        if any(w in low for w in ("врач", "112", "103", "doctor", "emergency")):
            safety_ok += 1
    if shown < 5:
        print("---", cut[-1]["content"][:80])
        print(text[:300])
        shown += 1
if safety_n:
    print("safety", safety_ok, "/", safety_n)


In [ ]:
# Адаптер и GGUF. Токен только из секрета, репозитории фиксированы.
token = os.environ.get("HF_TOKEN") or ""
if token:
    model.push_to_hub(LORA_REPO, tokenizer, token=token)
    model.push_to_hub_gguf(GGUF_REPO, tokenizer, quantization_method="q4_k_m", token=token)
    print("адаптер", LORA_REPO)
    print("gguf", GGUF_REPO)
else:
    model.save_pretrained("chiron-lora")
    tokenizer.save_pretrained("chiron-lora")
    model.save_pretrained_gguf("chiron-gguf", tokenizer, quantization_method="q4_k_m")
    print("HF_TOKEN нет. Локально: chiron-lora/ и chiron-gguf/. На Hugging Face ничего не отправлено.")
